# Addis Ababa Ride Demand Forecasting
## 02 — Data Analysis Report (Deliverable B)

All analysis uses the cleaned, joined `master_train.csv` from notebook 01 (all times in Addis Ababa local time, EAT).
Each task is under its own ID heading with a number/table and a short interpretation.

**Demand ratio** (used in B2/B3): `sum(trips) / sum(baseline)`, where *baseline* = mean trips for the same **zone × day-of-week × hour**
in "ordinary" hours (dry, no event window, no holiday/school break) × a zone-specific linear growth trend. A ratio of 1.0 = a normal hour.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
np.random.seed(42)

ROOT = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC, RAW = ROOT / "data" / "processed", ROOT / "data" / "raw"

m = pd.read_csv(PROC / "master_train.csv", parse_dates=["datetime"])
ev = pd.read_csv(PROC / "events_processed.csv", parse_dates=["start", "end"])
ev["zones"] = ev["zones"].fillna("").str.split("|")
weather = pd.read_csv(PROC / "weather_hourly_processed.csv", parse_dates=["datetime"])
m["date"] = m["datetime"].dt.normalize()

# Zone types from the shape of the weekday hourly profile (see B1.2 for the evidence table)
ZONE_TYPE = {
    "Arat Kilo": "business/CBD", "Kazanchis": "business/CBD", "Piassa": "business/CBD",
    "Ayat": "residential", "CMC": "residential", "Gerji": "residential", "Kolfe": "residential", "Sarbet": "residential",
    "Lideta": "transport hub", "Megenagna": "transport hub",
    "Merkato": "market",
    "Bole": "nightlife/airport",
}
m["zone_type"] = m["zone"].map(ZONE_TYPE)

# Ordinary-hour baseline -> demand ratio
ordinary = (m.rain_mm == 0) & (m.event_in_window == 0) & (m.is_public_holiday == 0) & (m.is_school_break == 0)
# baseline = typical shape (zone x weekday x hour) x zone-specific linear growth trend, both fitted on ordinary hours
KEYS = ["zone", "day_of_week", "hour"]
shape_ = m[ordinary].groupby(KEYS)["trips"].mean().rename("shape")
m = m.merge(shape_, on=KEYS, how="left")
level = {}
for z, g in m[ordinary].groupby("zone"):
    dly = g.groupby("date").agg(t=("trips", "sum"), s=("shape", "sum"))
    x = (dly.index - dly.index.min()).days.values
    level[z] = (np.polyfit(x, dly.t / dly.s, 1), dly.index.min())
m["level"] = [np.polyval(level[z][0], (d - level[z][1]).days) for z, d in zip(m.zone, m.date)]
m["baseline"] = m["shape"] * m["level"]
m["ratio"] = m["trips"] / m["baseline"].clip(lower=0.5)

def agg_ratio(df, by):
    """Demand ratio of a group = sum(trips) / sum(baseline) (robust to tiny night-time baselines)."""
    g = df.groupby(by)[["trips", "baseline"]].sum()
    return g.trips / g.baseline

print(m.shape, "| ordinary hours:", int(ordinary.sum()))
print("Overall ratio on ordinary hours (should be ~1):", round(m[ordinary].trips.sum() / m[ordinary].baseline.sum(), 3))

(82979, 43) | ordinary hours: 55734
Overall ratio on ordinary hours (should be ~1): 0.999


## B1 — Demand patterns

### B1.1 — Volume by zone

In [2]:
first = m.groupby("zone")["datetime"].min()
vol = m.groupby("zone").agg(total_trips=("trips", "sum"), zone_hours=("trips", "size"),
                            mean_per_zone_hour=("trips", "mean")).assign(first_hour=first)
vol["share_%"] = 100 * vol.total_trips / vol.total_trips.sum()
vol = vol.sort_values("total_trips", ascending=False).round(2)
display(vol)
print("City total trips:", int(vol.total_trips.sum()))

,total_trips,zone_hours,mean_per_zone_hour,first_hour,share_%
zone,,,,,
Merkato,285318.0,7043,40.51,2025-01-01,12.09
Bole,277213.0,7066,39.23,2025-01-01,11.74
Megenagna,269486.0,7050,38.22,2025-01-01,11.42
Kazanchis,224026.0,7058,31.74,2025-01-01,9.49
Piassa,199680.0,7069,28.25,2025-01-01,8.46
Lideta,197735.0,7060,28.01,2025-01-01,8.38
CMC,181962.0,7074,25.72,2025-01-01,7.71
Gerji,170335.0,7043,24.19,2025-01-01,7.22
Kolfe,162174.0,7061,22.97,2025-01-01,6.87


City total trips: 2360453


**Interpretation:** Merkato (12.1%), Bole (11.7%) and Megenagna (11.4%) carry over a third of all 2.36 M trips; Arat Kilo and Ayat are the smallest. Ayat only launched on 15 Mar 2025, so its 4.0% share understates it — compare zones on **mean trips per zone-hour** (Ayat 17.8 vs Merkato 40.5), not totals.

### B1.2 — Hour-of-day profile by zone type

In [3]:
wd = m[(m.is_weekend == 0) & (m.is_public_holiday == 0)]
prof = wd.pivot_table(index="zone", columns="hour", values="trips", aggfunc="mean")
shape = prof.div(prof.mean(axis=1), axis=0).round(1)            # normalised shape -> grouping evidence
shape.insert(0, "zone_type", shape.index.map(ZONE_TYPE))
display(shape.sort_values("zone_type"))

tp = wd.groupby(["zone_type", "hour"])["trips"].mean().unstack()
summary = pd.DataFrame({
    "peak_hour": tp.idxmax(axis=1), "peak_mean_trips": tp.max(axis=1).round(1),
    "2nd_peak_hour (other half of day)": [ (tp.loc[z, 12:].idxmax() if tp.loc[z].idxmax() < 12 else tp.loc[z, :11].idxmax()) for z in tp.index],
    "quietest_hour": tp.idxmin(axis=1), "quietest_mean_trips": tp.min(axis=1).round(1)})
display(summary)

hour,zone_type,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
zone,,,,,,,,,,,,,,,,,,,,,,,,,
Arat Kilo,business/CBD,0.1,0.1,0.1,0.1,0.1,0.2,0.6,1.4,2.0,1.5,0.7,0.6,1.0,1.4,1.2,1.1,1.6,2.4,2.6,2.1,1.4,0.9,0.4,0.2
Kazanchis,business/CBD,0.1,0.1,0.1,0.1,0.1,0.2,0.6,1.5,2.0,1.4,0.6,0.6,1.0,1.3,1.2,1.1,1.7,2.6,2.6,2.1,1.4,0.9,0.4,0.2
Piassa,business/CBD,0.1,0.1,0.1,0.1,0.1,0.2,0.6,1.5,2.0,1.5,0.7,0.6,1.0,1.3,1.2,1.2,1.7,2.4,2.6,2.1,1.5,0.8,0.5,0.2
Merkato,market,0.1,0.1,0.1,0.1,0.2,0.4,1.0,1.7,1.7,1.6,1.9,2.0,2.1,2.1,2.0,2.0,1.7,1.2,0.8,0.5,0.3,0.1,0.1,0.1
Bole,nightlife/airport,0.4,0.3,0.3,0.4,0.8,1.4,1.7,1.5,1.2,1.0,0.9,0.9,1.0,1.1,1.1,0.9,0.8,1.0,1.3,1.5,1.5,1.3,1.1,0.8
Ayat,residential,0.2,0.2,0.2,0.2,0.3,0.8,1.6,2.1,1.7,1.0,0.8,0.9,0.9,0.9,0.8,0.8,1.0,1.5,1.9,2.1,1.7,1.3,0.8,0.4
CMC,residential,0.2,0.2,0.2,0.2,0.3,0.8,1.6,2.2,1.8,1.1,0.8,0.8,0.9,0.8,0.8,0.7,1.0,1.5,1.9,2.0,1.7,1.3,0.8,0.4
Gerji,residential,0.2,0.2,0.2,0.2,0.3,0.8,1.6,2.1,1.8,1.0,0.8,0.8,0.9,0.9,0.8,0.8,1.0,1.4,1.9,2.0,1.7,1.3,0.9,0.4
Kolfe,residential,0.2,0.2,0.2,0.2,0.3,0.8,1.6,2.2,1.7,1.0,0.8,0.8,0.9,0.9,0.8,0.8,1.0,1.5,1.9,1.9,1.7,1.3,0.8,0.4


,peak_hour,peak_mean_trips,2nd_peak_hour (other half of day),quietest_hour,quietest_mean_trips
zone_type,,,,,
business/CBD,18,82.8,8,2,3.9
market,13,91.2,11,0,4.2
nightlife/airport,6,59.1,20,2,9.7
residential,7,48.2,19,2,3.7
transport hub,18,79.9,8,0,6.6


**Interpretation:** Five clear shapes: **business/CBD** (Arat Kilo, Kazanchis, Piassa) and **transport hubs** (Lideta, Megenagna) have twin commuter peaks at 08:00 and 18:00 with the evening one strongest; **residential** zones (Ayat, CMC, Gerji, Kolfe, Sarbet) peak earlier in the morning (07:00, outbound) and at 19:00 (home-bound); **Merkato** is a single broad midday plateau peaking at 13:00 and dead after 20:00; **Bole** (airport/nightlife) has a 06:00 airport peak, an evening peak at 20:00 and the highest night-time floor. All groups are quietest at 00:00–02:00.

### B1.3 — Weekday vs weekend

In [4]:
r = m[m.is_public_holiday == 0].groupby(["zone", "is_weekend"])["trips"].mean().unstack()
r.columns = ["weekday_mean", "weekend_mean"]
r["weekend_to_weekday"] = (r.weekend_mean / r.weekday_mean).round(2)
r["zone_type"] = r.index.map(ZONE_TYPE)
display(r.sort_values("weekend_to_weekday").round(2))

for z in ["Bole", "Kazanchis"]:
    dow = m[(m.zone == z) & (m.is_public_holiday == 0)].groupby("day_of_week")["trips"].mean()
    dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    print(f"{z} – mean trips by day of week:"); display(dow.round(1).to_frame().T)

,weekday_mean,weekend_mean,weekend_to_weekday,zone_type
zone,,,,
Arat Kilo,24.53,11.59,0.47,business/CBD
Piassa,33.83,15.96,0.47,business/CBD
Kazanchis,37.88,18.32,0.48,business/CBD
Megenagna,40.67,32.89,0.81,transport hub
Lideta,29.77,24.29,0.82,transport hub
Merkato,42.83,37.65,0.88,market
Ayat,17.54,18.46,1.05,residential
CMC,25.11,27.21,1.08,residential
Kolfe,22.47,24.17,1.08,residential


Bole – mean trips by day of week:


,Mon,Tue,Wed,Thu,Fri,Sat,Sun
trips,34.3,34.4,33.9,34.0,41.1,49.7,45.7


Kazanchis – mean trips by day of week:


,Mon,Tue,Wed,Thu,Fri,Sat,Sun
trips,37.8,37.2,37.7,37.0,39.6,18.3,18.4


**Interpretation:** Business/CBD zones collapse at weekends (ratio 0.47–0.48 — demand halves), transport hubs and Merkato dip moderately (0.81–0.88), residential zones are slightly busier (1.05–1.09) and Bole is the clear weekend winner (1.34). Bole's day-of-week shape builds from ~34 trips Mon–Thu to 41 on Friday and peaks at 49.7 on Saturday, consistent with nightlife and leisure travel.

### B1.4 — Trend January → October

In [5]:
# Full weeks only, and only the 11 zones open all year so Ayat's launch does not inflate growth
core = m[m.zone != "Ayat"]
wk = core.set_index("datetime")["trips"].resample("W-SUN").sum()
cnt = core.set_index("datetime")["trips"].resample("W-SUN").size()
wk = wk[cnt >= 0.95 * cnt.median()]                  # drop partial / outage weeks
x = np.arange(len(wk)); slope, icpt = np.polyfit(x, wk.values, 1)
first4, last4 = wk.iloc[:4].mean(), wk.iloc[-4:].mean()
print(f"Weekly trips (11 core zones): first 4 weeks {first4:,.0f} -> last 4 weeks {last4:,.0f}  "
      f"growth {100*(last4/first4-1):.1f}%")
print(f"Linear trend: +{slope:,.0f} trips/week per week  (= {100*slope/wk.mean():.2f}% of mean weekly volume)")
monthly = core.groupby("month")["trips"].mean().round(2)
display(monthly.to_frame("mean trips per zone-hour (core zones)").T)
proj = icpt + slope * (len(wk) + 1.5)
print(f"Extrapolated weekly volume for early November ≈ {proj:,.0f} vs October mean {wk.iloc[-4:].mean():,.0f}")

Weekly trips (11 core zones): first 4 weeks 43,190 -> last 4 weeks 60,275  growth 39.6%
Linear trend: +503 trips/week per week  (= 0.96% of mean weekly volume)


month,1,2,3,4,5,6,7,8,9,10
mean trips per zone-hour (core zones),24.11,24.9,26.2,27.31,28.1,29.25,31.66,32.18,33.7,34.0


Extrapolated weekly volume for early November ≈ 63,723 vs October mean 60,275


**Interpretation:** Demand grows steadily: weekly trips in the 11 always-open zones rose from ~43,200 to ~60,300 (+39.6%), a linear trend of ≈ +500 trips/week each week (~1% per week); mean trips per zone-hour climbs every month from 24.1 (Jan) to 34.0 (Oct). A model trained on raw history would under-forecast November, so it needs a trend feature (e.g. `days_since_start`) or recent-level lags; extrapolation suggests early-November weekly volume ≈ 63,700, ~5% above the October average.

## B2 — Weather

### B2.1 — Time-zone check

In [6]:
# (a) Temperature curve: raw file clock (UTC) vs corrected EAT clock
t = weather[weather.data_type == "observed"]
curve = pd.DataFrame({"EAT_clock": t.groupby(t.datetime.dt.hour)["temp_c"].mean(),
                      "raw_UTC_clock": t.groupby((t.datetime - pd.Timedelta(hours=3)).dt.hour)["temp_c"].mean()}).round(1)
print("Temperature peak hour: raw clock", curve.raw_UTC_clock.idxmax(), "| EAT clock", curve.EAT_clock.idxmax(),
      "| minimum EAT", curve.EAT_clock.idxmin())

# (b) Rain–demand relationship as the weather clock is mis-aligned by k hours (k=0 is our corrected join)
city = m.groupby("datetime").agg(trips=("trips", "sum"), baseline=("baseline", "sum"))
city["ratio"] = city.trips / city.baseline
rain = weather.set_index("datetime")["rain_mm"]
rows = []
for k in range(-3, 6):
    shifted = rain.copy(); shifted.index = shifted.index + pd.Timedelta(hours=k)
    j = city.join(shifted, how="inner")
    rainy = j.rain_mm > 0
    rows.append({"shift_h": k, "note": "raw UTC (no correction)" if k == -3 else ("corrected EAT" if k == 0 else ""),
                 "corr(rain, demand ratio)": round(j.ratio.corr(j.rain_mm), 3),
                 "ratio rainy hours": round(j.ratio[rainy].mean(), 3), "ratio dry hours": round(j.ratio[~rainy].mean(), 3)})
shift_tbl = pd.DataFrame(rows)
display(shift_tbl)

Temperature peak hour: raw clock 12 | EAT clock 15 | minimum EAT 3


,shift_h,note,"corr(rain, demand ratio)",ratio rainy hours,ratio dry hours
0,-3,raw UTC (no correction),0.056,1.065,1.011
1,-2,,0.083,1.081,1.010
2,-1,,0.103,1.093,1.009
3,0,corrected EAT,0.420,1.212,0.999
4,1,,0.101,1.097,1.009
5,2,,0.115,1.093,1.009
6,3,,0.094,1.081,1.010
7,4,,0.105,1.082,1.010
8,5,,0.082,1.076,1.010


**Interpretation:** The weather file is on **UTC**: on the raw clock the temperature peaks at 12:00 (solar noon), after +3 h it peaks at 15:00 EAT with the minimum at 03:00 — a normal diurnal cycle. The rain–demand correlation confirms it: it is 0.42 at the corrected alignment and collapses to 0.06–0.12 for any other shift (0.056 with no correction), and the rainy-hour uplift drops from +21% to ~+6–9%. Joining the raw clock would hide most of the weather signal.

### B2.2 — Rain effect by zone type

In [7]:
# Ratio is already "vs the same zone, weekday, hour (and month) in dry ordinary hours"
d = m[(m.event_in_window == 0) & (m.is_public_holiday == 0)]
eff = agg_ratio(d.assign(rainy=d.rain_mm > 0), ["zone_type", "rainy"]).unstack()
eff.columns = ["dry", "rainy"]
eff["rain_uplift_%"] = 100 * (eff.rainy / eff.dry - 1)
eff["rainy_zone_hours"] = d[d.rain_mm > 0].groupby("zone_type").size()
display(eff.round(3).sort_values("rain_uplift_%", ascending=False))

,dry,rainy,rain_uplift_%,rainy_zone_hours
zone_type,,,,
residential,0.986,1.276,29.360,2535
nightlife/airport,1.002,1.293,29.125,440
transport hub,0.991,1.269,28.059,1031
business/CBD,0.997,1.264,26.824,1489
market,1.004,0.789,-21.421,515


**Interpretation:** Rain raises demand by ~27–29% in every zone type **except Merkato, where it falls by ~21%** — an open-air market loses shoppers when it rains, while elsewhere rain converts walkers and minibus riders into ride-hail customers. So rain does not increase demand everywhere, and a model needs a rain × zone interaction (trees will learn it from `zone` + `rain_mm`).

### B2.3 — Rain dose-response

In [8]:
order = ["none", "light", "moderate", "heavy"]
dr = agg_ratio(d, ["zone_type", "rain_class"]).unstack()[order]
allz = agg_ratio(d, "rain_class")[order]
dr.loc["ALL ZONES"] = allz
dr = dr.div(dr["none"], axis=0)
display(dr.round(3))
print("Rainy zone-hours per class:", d.rain_class.value_counts()[order].to_dict())
print("Rain class bins (mm/h): none=0, light (0,2.5], moderate (2.5,7.6], heavy >7.6")

rain_class,none,light,moderate,heavy
zone_type,,,,
business/CBD,1.0,1.154,1.366,1.574
market,1.0,0.866,0.723,0.569
nightlife/airport,1.0,1.171,1.369,1.655
residential,1.0,1.176,1.381,1.627
transport hub,1.0,1.167,1.386,1.543
ALL ZONES,1.0,1.129,1.297,1.459


Rainy zone-hours per class: {'none': 71570, 'light': 3382, 'moderate': 1981, 'heavy': 647}
Rain class bins (mm/h): none=0, light (0,2.5], moderate (2.5,7.6], heavy >7.6


**Interpretation:** The response is monotonic but **sub-linear**: city-wide light rain (≤2.5 mm/h) gives +13%, moderate +30%, heavy +46%, so each extra millimetre adds less — the effect begins to saturate in heavy rain, plausibly because driver supply caps the trips served. Merkato mirrors the pattern downward (0.87 → 0.72 → 0.57).

## B3 — Events & calendar

### B3.1 — Public holidays

In [9]:
daily = m.groupby(["date", "zone"])["trips"].sum().unstack()
city_d = daily.sum(axis=1)
hol = ev[(ev.event_type == "public_holiday") & (ev.status != "cancelled")]
hol_days = sorted({d for _, e in hol.iterrows() for d in pd.date_range(e.start.normalize(), e.end.normalize())
                   if d in city_d.index})
hol_set = set(hol_days)
name_of = {d: e.event_name for _, e in hol.iterrows() for d in pd.date_range(e.start.normalize(), e.end.normalize())}
rows, zone_idx = [], {}
for dday in hol_days:
    refs = [dday + pd.Timedelta(weeks=w) for w in (-2, -1, 1, 2)]
    refs = [r for r in refs if r in city_d.index and r not in hol_set]
    rows.append({"date": dday.date(), "holiday": name_of[dday], "weekday": dday.day_name(),
                 "city_trips": int(city_d[dday]), "ref_mean": round(city_d[refs].mean()),
                 "index_vs_normal": round(city_d[dday] / city_d[refs].mean(), 3)})
    zone_idx[dday.date()] = (daily.loc[dday] / daily.loc[refs].mean()).round(2)
hol_tbl = pd.DataFrame(rows).sort_values("index_vs_normal")
display(hol_tbl)
zi = pd.DataFrame(zone_idx).T
print("Zone-level index (holiday / same weekday ±1–2 weeks):"); display(zi)

,date,holiday,weekday,city_trips,ref_mean,index_vs_normal
4,2025-04-18,Good Friday,Friday,6056,8164,0.742
0,2025-01-07,Genna (Ethiopian Christmas),Tuesday,4805,6318,0.761
9,2025-06-06,Eid al-Adha,Friday,6806,8660,0.786
10,2025-09-04,Mawlid,Thursday,7414,9124,0.813
11,2025-09-11,Enkutatash (Ethiopian New Year),Thursday,7783,9341,0.833
8,2025-05-28,Downfall of the Derg,Wednesday,6754,8024,0.842
3,2025-03-31,Eid al-Fitr,Monday,6275,7383,0.850
6,2025-05-01,International Labour Day,Thursday,6611,7524,0.879
12,2025-09-27,Meskel (Finding of the True Cross),Saturday,8259,8734,0.946
5,2025-04-20,Fasika (Ethiopian Easter),Sunday,6059,6251,0.969


Zone-level index (holiday / same weekday ±1–2 weeks):


zone,Arat Kilo,Ayat,Bole,CMC,Gerji,Kazanchis,Kolfe,Lideta,Megenagna,Merkato,Piassa,Sarbet
2025-01-07,0.41,NaN,1.21,1.12,0.98,0.46,1.18,0.75,0.76,0.50,0.41,1.00
2025-01-19,1.27,NaN,0.99,0.95,1.00,0.90,0.98,1.03,1.05,0.95,1.26,1.02
2025-03-02,1.10,NaN,1.09,1.05,1.02,1.39,1.03,0.95,1.12,1.08,1.32,1.05
2025-03-31,0.53,1.20,1.34,1.03,1.19,0.51,1.12,0.76,0.87,0.60,0.49,1.17
2025-04-18,0.44,1.04,0.98,1.00,0.88,0.43,1.00,0.77,0.83,0.52,0.46,0.97
2025-04-20,0.81,0.97,1.07,0.79,0.94,0.99,1.04,0.97,1.05,1.01,0.93,0.88
2025-05-01,0.49,1.13,1.38,1.14,1.18,0.49,0.97,0.88,0.94,0.55,0.50,1.28
2025-05-05,0.61,1.55,1.66,1.38,1.45,0.64,1.36,1.15,1.12,0.58,0.49,1.34
2025-05-28,0.58,1.07,1.25,1.10,1.19,0.45,1.09,0.87,0.76,0.65,0.48,1.02
2025-06-06,0.52,1.08,1.17,1.05,1.03,0.58,1.12,0.71,0.69,0.47,0.44,1.00


**Interpretation:** Most holidays **reduce** city demand: Good Friday (index 0.74), Genna (0.76) and Eid al-Adha (0.79) are the biggest drops, while Sunday holidays (Timkat, Adwa) and Patriots' Victory Day are near or slightly above normal (1.01–1.08). The zone table shows the reaction is local: business zones (Arat Kilo, Kazanchis, Piassa) and Merkato fall to ~0.4–0.6 on almost every holiday, whereas Bole and the residential zones often rise (up to 1.3–1.7, e.g. 5 May) as people visit family and go out. On Meskel (27 Sep) Kazanchis and Piassa rise (1.39 / 1.18) because the Meskel Square celebrations are next to them.

### B3.2 — Event-window study (football)

In [10]:
def window_effect(etype):
    out = []
    for _, e in ev[(ev.event_type == etype) & (ev.status != "cancelled")].iterrows():
        s, en = e.start.floor("h"), e.end.ceil("h")
        wins = {"pre (2h before)": pd.date_range(s - pd.Timedelta(hours=2), s - pd.Timedelta(hours=1), freq="h"),
                "during": pd.date_range(s, en - pd.Timedelta(hours=1), freq="h"),
                "post (2h after)": pd.date_range(en, en + pd.Timedelta(hours=1), freq="h")}
        for z in e.zones:
            for w, hrs in wins.items():
                sub = m[(m.zone == z) & m.datetime.isin(hrs)]
                if len(sub):
                    out.append({"event_id": e.event_id, "zone": z, "window": w,
                                "trips": sub.trips.sum(), "baseline": sub.baseline.sum()})
    return pd.DataFrame(out)

fb = window_effect("football_match")
res = fb.groupby("window")[["trips", "baseline"]].sum()
res["uplift_%"] = 100 * (res.trips / res.baseline - 1)
res["n_matches"] = fb.groupby("window").event_id.nunique()
display(res.loc[["pre (2h before)", "during", "post (2h after)"]].round(1))
print("Baseline = same zone, weekday and hour in ordinary hours, scaled by the zone growth trend.")

,trips,baseline,uplift_%,n_matches
window,,,,
pre (2h before),5603.0,3207.5,74.7,41
during,3789.0,2793.0,35.7,42
post (2h after),5112.0,2287.5,123.5,43


Baseline = same zone, weekday and hour in ordinary hours, scaled by the zone growth trend.


**Interpretation:** All three windows show strong uplift in the match zone vs the same ordinary zone-hours, but the **2 hours after the final whistle are largest (+124%)**, followed by the 2 hours before kick-off (+75%) and the match itself (+36%, when fans are inside the stadium). This justifies modeling separate pre/during/post flags and extending event windows past the end time.

### B3.3 — Event type ranking

In [11]:
rows = []
for et in ["football_match", "concert", "conference", "exhibition", "road_closure", "sports_run"]:
    w = window_effect(et)
    if w.empty: continue
    g = w.groupby("window")[["trips", "baseline"]].sum()
    eff_w = (g.trips / g.baseline - 1) * 100
    per_event = w.groupby("event_id")[["trips", "baseline"]].sum()
    pe = per_event.trips / per_event.baseline - 1
    se = pe.std(ddof=1) / np.sqrt(len(pe)) * 100 if len(pe) > 1 else np.nan
    rows.append({"event_type": et, "n_events": w.event_id.nunique(),
                 "pre_%": eff_w.get("pre (2h before)"), "during_%": eff_w.get("during"), "post_%": eff_w.get("post (2h after)"),
                 "whole_window_%": 100 * (g.trips.sum() / g.baseline.sum() - 1), "±1.96·SE_%": 1.96 * se})
rank = pd.DataFrame(rows).set_index("event_type")
rank["measurable?"] = np.where(rank["whole_window_%"].abs() > rank["±1.96·SE_%"], "yes", "no")
display(rank.sort_values("whole_window_%", ascending=False).round(1))

,n_events,pre_%,during_%,post_%,whole_window_%,±1.96·SE_%,measurable?
event_type,,,,,,,
football_match,43,74.7,35.7,123.5,75.0,17.3,yes
concert,25,53.9,3.1,84.6,30.8,5.2,yes
conference,37,15.5,14.1,32.6,18.9,6.8,yes
exhibition,7,0.2,3.0,-5.0,2.8,4.3,no
sports_run,2,20.6,-35.4,10.5,-11.8,14.2,no
road_closure,22,5.8,-31.3,0.3,-23.5,3.5,yes


**Interpretation:** Ranked by whole-window effect: football (+75%), concerts (+31%, again peaking after the show at +85%) and conferences (+19%) have significant positive effects; road closures significantly **reduce** demand (−24%, −31% while active). Exhibitions (+3%) and sports runs (−12% with only 2 events, CI ±14%) have **no measurable effect**, so a separate feature for them would only add noise.

### B3.4 — Cancelled and unlisted events

In [12]:
# (a) cancelled events: same window study but for status == cancelled
canc = ev[ev.status == "cancelled"]
rows = []
for _, e in canc.iterrows():
    hrs = pd.date_range(e.start.floor("h") - pd.Timedelta(hours=2), e.end.ceil("h") + pd.Timedelta(hours=1), freq="h")
    sub = m[m.zone.isin(e.zones) & m.datetime.isin(hrs)]
    rows.append({"event_id": e.event_id, "event": e.event_name, "type": e.event_type, "zones": ", ".join(e.zones),
                 "start": e.start, "zone_hours": len(sub),
                 "uplift_%": round(100 * (sub.trips.sum() / sub.baseline.sum() - 1), 1) if len(sub) else np.nan})
print("(a) Cancelled events:"); display(pd.DataFrame(rows))

# (b) largest zone-day spikes NOT inside any listed event window or holiday
zd = m.groupby(["zone", "date"]).agg(trips=("trips", "sum"), baseline=("baseline", "sum"),
                                     ev=("event_in_window", "max"), hol=("is_public_holiday", "max"), sb=("is_school_break", "max"), n=("trips", "size")).reset_index()
zd["ratio"] = zd.trips / zd.baseline
unl = zd[(zd.ev == 0) & (zd.hol == 0) & (zd.sb == 0) & (zd.n >= 22)].nlargest(8, "ratio")
unl["weekday"] = unl.date.dt.day_name()
print("(b) Largest unexplained zone-day spikes:"); display(unl.round(2))

(a) Cancelled events:


,event_id,event,type,zones,start,zone_hours,uplift_%
0,EVT-0113,Live concert,concert,Bole,2025-09-01 20:00:00,8,16.0
1,EVT-0052,Premier league match,football_match,Kazanchis,2025-04-23 18:00:00,6,16.7
2,EVT-0058,Premier league match,football_match,Kazanchis,2025-05-03 15:00:00,6,-6.6
3,EVT-0042,Conference: Health congress,conference,Kazanchis,2025-04-03 08:00:00,11,-9.4
4,EVT-0099,Live concert,concert,Bole,2025-07-28 20:00:00,8,12.6
5,EVT-0020,Premier league match,football_match,Kazanchis,2025-02-22 15:00:00,6,74.8


(b) Largest unexplained zone-day spikes:


,zone,date,trips,baseline,ev,hol,sb,n,ratio,weekday
1715,Kazanchis,2025-09-26,1761.0,1011.30,0.0,0,0,23,1.74,Friday
2771,Merkato,2025-04-19,1804.0,1223.64,0.0,0,0,23,1.47,Saturday
410,Ayat,2025-06-29,554.0,392.34,0.0,0,0,23,1.41,Sunday
352,Ayat,2025-05-02,448.0,324.97,0.0,0,0,23,1.38,Friday
2915,Merkato,2025-09-10,1605.0,1178.83,0.0,0,0,24,1.36,Wednesday
349,Ayat,2025-04-29,438.0,324.23,0.0,0,0,23,1.35,Tuesday
3052,Piassa,2025-03-27,938.0,709.73,0.0,0,0,24,1.32,Thursday
350,Ayat,2025-04-30,432.0,331.35,0.0,0,0,24,1.30,Wednesday


**Interpretation:** (a) Five of the six cancelled events leave no meaningful footprint (−9% to +17%, within normal hour-to-hour noise), confirming they should be excluded. The exception, the Kazanchis football match of 22 Feb (+75%), behaves like a match that actually happened — likely a mislabelled status or a rescheduled game, worth flagging. (b) Large unexplained spikes: **Kazanchis 26 Sep (×1.74)** = Meskel eve (Demera bonfire at Meskel Square), **Merkato 19 Apr (×1.47)** = Saturday before Fasika (Easter shopping), **Merkato 10 Sep (×1.36)** = eve of Enkutatash (New Year shopping). The Ayat spikes in April–June are early post-launch volatility. Holiday *eves* are missing from the calendar and could be added as a feature.

## B4 — Operations & data quality

### B4.1 — Operational variables vs demand

In [13]:
ops = m[["trips", "active_drivers", "avg_wait_min", "avg_fare_birr"]]
display(ops.corr(method="pearson").loc[["trips"]].round(3))
display(ops.corr(method="spearman").loc[["trips"]].round(3).rename(index={"trips": "trips (Spearman)"}))
print("Median trips per active driver-hour:", round((m.trips / m.active_drivers).median(), 2))

,trips,active_drivers,avg_wait_min,avg_fare_birr
trips,1.0,0.987,0.534,0.006


,trips,active_drivers,avg_wait_min,avg_fare_birr
trips (Spearman),1.0,0.993,0.639,0.072


Median trips per active driver-hour: 1.31


**Interpretation:** Trips correlate almost perfectly with active drivers (r = 0.99, ~1.31 trips per driver-hour), moderately with wait time (0.53) and not at all with fare (0.01). These are **consequences** of demand: drivers log on and waits lengthen *because* riders are requesting, and fares are set per zone. None of them is known for next Tuesday at 18:00, so using them as inputs would leak the target; they are excluded from the model (D4) but used for the demo's driver and revenue estimates.

### B4.2 — Gaps and outages

In [14]:
TRAIN_START, TRAIN_END = pd.Timestamp("2025-01-01"), pd.Timestamp("2025-10-31 23:00")
hours = pd.date_range(TRAIN_START, TRAIN_END, freq="h")
zones = sorted(m.zone.unique())
present_raw_hours = m.groupby("datetime").size().reindex(hours, fill_value=0)

grid = pd.MultiIndex.from_product([zones, hours], names=["zone", "datetime"]).to_frame(index=False)
grid = grid.merge(first.rename("first_hour").reset_index(), on="zone")
grid = grid.merge(m[["zone", "datetime"]].assign(in_master=True), how="left", on=["zone", "datetime"])
grid["in_master"] = grid["in_master"].fillna(False).astype(bool)
grid["kind"] = np.select(
    [grid.in_master.to_numpy(), (grid.datetime < grid.first_hour).to_numpy(), grid.datetime.map(present_raw_hours).eq(0).to_numpy()],
    ["present", "zone not launched (Ayat)", "platform outage"], "random missing / invalid record")
display(grid.kind.value_counts().to_frame("zone_hours"))

out_hours = present_raw_hours[present_raw_hours == 0].index
blocks = pd.Series(out_hours).diff().ne(pd.Timedelta(hours=1)).cumsum()
outages = pd.Series(out_hours).groupby(blocks.values).agg(["min", "max", "size"])
outages.columns = ["start", "end", "hours"]
print("City-wide outage periods (no zone has any row):"); display(outages)
print("Ayat first hour:", first["Ayat"])
rnd = grid[grid.kind == "random missing / invalid record"]
print("Random missing by zone:"); display(rnd.zone.value_counts().to_frame("zone_hours").T)

,zone_hours
kind,
present,82979
random missing / invalid record,2317
zone not launched (Ayat),1752
platform outage,504


City-wide outage periods (no zone has any row):


,start,end,hours
1,2025-05-12 02:00:00,2025-05-13 19:00:00,42


Ayat first hour: 2025-03-15 00:00:00
Random missing by zone:


zone,Gerji,Merkato,Megenagna,Sarbet,Arat Kilo,Kazanchis,Lideta,Kolfe,Bole,Piassa,CMC,Ayat
zone_hours,211,211,204,203,197,196,194,193,188,185,180,155


**Interpretation:** Of 87,552 possible zone-hours, 82,979 are usable. There are three different kinds of gap: (1) one **platform outage** from 12 May 02:00 to 13 May 19:00 (42 h, 504 zone-hours) where no zone has any row — excluded, not imputed; (2) **Ayat not yet launched** before 15 Mar (1,752 zone-hours) — excluded, they are not zero-demand hours; (3) **2,317 random missing or invalid records** (dropped rows, −1 sentinels, impossible spikes) spread evenly across zones (155–211 each) — excluded from the target and handled by lag features' NaN tolerance.

### B4.3 — Pay-period effect

In [15]:
# Detrend with the ordinary-hour baseline WITHOUT month (so payday days aren't absorbed), then compare
k2 = ["zone", "day_of_week", "hour"]
dd = m[(m.is_public_holiday == 0) & (m.event_in_window == 0)].copy()
dd["week"] = dd.datetime.dt.to_period("W")
wk_level = dd.groupby(["zone", "week"])["trips"].transform("mean")            # local level (removes trend)
dd["resid_ratio"] = dd.trips / (dd.groupby(k2)["trips"].transform("mean") * wk_level / dd.groupby("zone")["trips"].transform("mean"))
dd["day_class"] = np.select([dd.day_of_month >= 25, dd.day_of_month <= 3], ["payday: 25th–end", "payday: 1st–3rd"], "ordinary")
pay = dd.groupby("day_class")["resid_ratio"].agg(["mean", "count", "std"])
pay["ci95"] = 1.96 * pay["std"] / np.sqrt(pay["count"])
pay["vs_ordinary_%"] = 100 * (pay["mean"] / pay.loc["ordinary", "mean"] - 1)
display(pay.round(4))
by_dom = dd.groupby("day_of_month")["resid_ratio"].mean()
display((100 * (by_dom / by_dom[(by_dom.index > 3) & (by_dom.index < 25)].mean() - 1)).round(1).to_frame("vs ordinary %").T)

,mean,count,std,ci95,vs_ordinary_%
day_class,,,,,
ordinary,0.9993,53544,0.3164,0.0027,0.0000
payday: 1st–3rd,1.0131,7510,0.3191,0.0072,1.3846
payday: 25th–end,1.0064,16526,0.3144,0.0048,0.7060


day_of_month,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
vs ordinary %,1.8,4.2,-1.6,-1.2,-1.3,1.4,-0.9,0.8,-0.0,0.8,0.1,2.3,1.9,0.1,-0.9,-0.5,-0.2,1.0,0.7,-0.0,-0.2,-1.8,0.9,-3.0,-3.0,2.9,1.4,-0.5,2.8,0.7,0.6


**Interpretation:** After removing each zone's weekly level and its weekday-hour shape, demand on the 1st–3rd is +1.4% (±0.7%) and on the 25th–end +0.7% (±0.5%) vs ordinary days; the day-of-month curve wobbles by ±2–3% with no clear payday bump. The effect is statistically detectable but tiny compared with weather (+13–46%) or events (+75%), so `is_payday_window` is kept only as a cheap optional feature and is expected to add little.